# Additional n=1 external validation prediction

to clean up and test

to check:

- only include ECoG and ipsilateral STN LFP
- exclude moments where was only Dyskinesia in body-side ipsilateral to ECoG (NOT CORRESPONDING WITH ECoG-hemisphere)

## Load packages and functions

In [ ]:
# Importing Python and external packages
import os
import importlib
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt


In [ ]:
def get_project_path_in_notebook(
    subfolder: str = '',
):
    """
    Finds path of projectfolder from Notebook.
    Start running this once to correctly find
    other modules/functions
    """
    path = os.getcwd()

    while path[-20:] != 'dyskinesia_neurophys':

        path = os.path.dirname(path)
    
    return path

In [ ]:
# define local storage directories
projectpath = get_project_path_in_notebook()
codepath = os.path.join(projectpath, 'code')
figpath = os.path.join(projectpath, 'figures')
datapath = os.path.join(projectpath, 'data')
feat_path = os.path.join(projectpath, 'results', 'features')

In [ ]:
os.chdir(codepath)
# own utility functions
import utils.utils_fileManagement as utilsFiles
# own data exploration functions
import lfpecog_features.feats_read_proc_data as read_data
import lfpecog_preproc.preproc_import_scores_annotations as importClin
import lfpecog_analysis.ft_processing_helpers as ftProc
import lfpecog_analysis.import_ephys_results as importResults
import lfpecog_analysis.stats_fts_lid_corrs as ftLidCorr
import lfpecog_analysis.load_SSD_features as load_ssdFts
import lfpecog_analysis.ft_processing_helpers as ftProc
import lfpecog_features.feats_helper_funcs as ftHelp
from lfpecog_features.get_ssd_data import get_subject_SSDs
import lfpecog_predict.prepare_predict_arrays as prep_pred_arrs

from lfpecog_plotting.plotHelpers import get_colors
import lfpecog_plotting.plotHelpers as pltHelp
import lfpecog_plotting.plot_FreqCorr as plotFtCorrs
import lfpecog_plotting.plot_SSD_feat_descriptives as plot_ssd_descr

import lfpecog_analysis.get_acc_task_derivs as accDerivs

## 1) Define data, feature settings, and create DataClasses

In [ ]:
# set variables
DATA_VERSION = 'v4.0'    # v4.0: new artef-rem, no reref; v3.0 multiple re-ref
FT_VERSION = 'v8'  # v4: broad-flanks, bursts; v3: broad-flanked SSD
INCL_PSD_FTS=['mean_psd', 'variation']
IGNORE_PTS = ['011', '104', '106']

CDRS_RATER = 'Patricia'
ANALYSIS_SIDE = 'BILAT'
INCL_CORE_CDRS = True
CATEG_CDRS = False
MILD_CDRS = 4
SEV_CDRS = 8

INCL_ECOG = False
INCL_ACC = True

# path were classes are saved and loaded from
extVal_path = os.path.join(utilsFiles.get_project_path('data'),
                           'ext_val_prediction')

for debugging single sub feat classes

In [ ]:
importlib.reload(load_ssdFts)


# get all available subs with features
SUBS = utilsFiles.get_avail_ssd_subs(DATA_VERSION=DATA_VERSION,
                                     FT_VERSION=FT_VERSION,
                                     IGNORE_PTS=IGNORE_PTS)
print(f'SUBS: n={len(SUBS)} ({SUBS})')

# use as single ft example to debug/develop
sub_fts = load_ssdFts.ssdFeatures(
    sub_list=['023', '024'],
    settings_json=f'ftExtr_spectral_{FT_VERSION}.json',
)

In [ ]:
sub_fts.sub024

Prepare Class with FEATS and CDRS-LABELS

In [ ]:
# LOAD FEATURE via FeatureClass containing all features
importlib.reload(utilsFiles)
importlib.reload(accDerivs)
importlib.reload(ftProc)
importlib.reload(importClin)
importlib.reload(load_ssdFts)
importlib.reload(ftLidCorr)


FeatLid = ftProc.FeatLidClass(
    FT_VERSION=FT_VERSION,
    CDRS_RATER=CDRS_RATER,
    INCL_ECOG=INCL_ECOG,
    INCL_ACC_RMS=INCL_ACC,
    CATEGORICAL_CDRS=CATEG_CDRS,
    CORR_TARGET='CDRS',
    cutMild=MILD_CDRS, cutSevere=SEV_CDRS,
    TO_CALC_CORR=False,
    verbose=True,
)

# print(f'features included: {FEATS[sub].keys()}') 

Save Classes

In [ ]:
# SAVE FeatLabelClass as pickle

className = f'featLabels_n{len(FeatLid.FEATS.keys())}_ft{FT_VERSION}'
if FeatLid.CORR_TARGET == 'LID': className += '_Lid'
elif FeatLid.CATEGORICAL_CDRS == True: className += '_CatCdrs'
else: className += '_Cdrs'

if FeatLid.INCL_ECOG: className += '_Ecog'
else: className += '_StnOnly'

if INCL_ACC: className += '_ACC'

utilsFiles.save_class_pickle(class_to_save=FeatLid,
                             path=extVal_path,
                             filename=className)

Import classes

In [ ]:
# LOAD existing classes with features and labels

if INCL_ECOG:
    fname_ext = '_Ecog'
    n_subs = 14
else:
    fname_ext = '_StnOnly'
    n_subs = 22

if INCL_ACC: fname_ext += '_ACC'

predData = utilsFiles.load_class_pickle(
    os.path.join(extVal_path,
                 f'featLabels_n{n_subs}_ft{FT_VERSION}_Cdrs{fname_ext}.P'),
    convert_float_np64=True
)


## 2) Prepare prediction arrays

Prepare X (features) and y (labels, LID) arrays for prediction analyses.

- 1) use all epochs, regardless of movement presence
- 2) make prediction movement aware, A) one classifier trains on and tests all epochs without (many) movements; B) one classifier trains on and tests epochs with more movement (movement-zscore-threshold is defined based on lineplot PPV/NPV)



Split training and external validation-test data


In [ ]:
dataDicts = {"train": {'FEATS': {}, 'LABELS': {}, 'ACC': {}},
             "extVal": {'FEATS': {}, 'LABELS': {}, 'ACC': {}}}

for sub in predData.FEATS.keys():
    if sub != '024':
        dataDicts['train']['FEATS'][sub] = predData.FEATS[sub]
        dataDicts['train']['LABELS'][sub] = predData.FT_LABELS[sub]
        dataDicts['train']['ACC'][sub] = predData.ACC_RMS[sub]

    elif sub == '024':
        dataDicts['extVal']['FEATS'][sub] = predData.FEATS[sub]
        dataDicts['extVal']['LABELS'][sub] = predData.FT_LABELS[sub]
        dataDicts['extVal']['ACC'][sub] = predData.ACC_RMS[sub]

print('n-train', len(dataDicts['train']['FEATS'].keys()),
      '; n-validate', len(dataDicts['extVal']['FEATS'].keys()))



In [ ]:

def merge_pred_dicts_to_grouparrays(dataDict, accDict=False,):

    list_returns = prep_pred_arrs.get_group_arrays_for_prediction(
        feat_dict=dataDict['FEATS'],
        label_dict=dataDict['LABELS'],
        CDRS_CODING='binary',  # categorical
        acc_dict=accDict,
    )
    if len(list_returns) == 6:
        (X_total, y_total_binary, y_total_scale,
         sub_ids_total, ft_times_total, ft_names) = list_returns
        acc_total = False  # set false bool bcs absent
    elif len(list_returns) == 7:
        (X_total, y_total_binary, y_total_scale,
         sub_ids_total, ft_times_total, ft_names,
         acc_total) = list_returns

    # Merge subject-arrays to one group array for prediction
    list_returns = prep_pred_arrs.merge_group_arrays(
        X_total=X_total,
        y_total_binary=y_total_binary,
        y_total_scale=y_total_scale,
        sub_ids_total=sub_ids_total,
        ft_times_total=ft_times_total,
        ext_acc_arr=acc_total
    )
    if len(list_returns) == 5:
        (X_all, y_all_binary, y_all_scale,
         sub_ids, ft_times_all) = list_returns
    elif len(list_returns) == 6:
        (X_all, y_all_binary, y_all_scale,
         sub_ids, ft_times_all, acc_total) = list_returns

    print(f'Subjects included ({len(np.unique(sub_ids))}): {np.unique(sub_ids)}')


    if accDict == False:
        return X_all, y_all_binary, sub_ids, ft_times_all, ft_names

    else:
        return X_all, y_all_binary, sub_ids, ft_times_all, ft_names, acc_total




In [ ]:
# Create arrays per subject based on features and labels

importlib.reload(prep_pred_arrs)

(X_all, y_all_binary,
 sub_ids, ft_times_all,
 ft_names, acc_all) = {}, {}, {}, {}, {}, {}

for label in dataDicts.keys():

    (
        X_all[label], y_all_binary[label],
        sub_ids[label], ft_times_all[label],
        ft_names[label], acc_all[label]
    ) = merge_pred_dicts_to_grouparrays(
        dataDicts[label], accDict=dataDicts[label]['ACC']
    )



In [ ]:
for label in dataDicts.keys():
    
    print(label, X_all[label].shape, y_all_binary[label].shape)

print(ft_names['extVal'])

In [ ]:
def convert_features(X_arr, ft_names):
    """
    takes average over bilat lfp features
    takes average over single gamma bands into gammaBroad
    """
    X_df = pd.DataFrame(X_arr, columns=ft_names)

    # change unilat into mean bilat LFP powers per band
    for band in ['theta', 'alpha', 'lo_beta', 'hi_beta',
                'gamma1', 'gamma2', 'gamma3', 'gammaPeak']:
        for ft in ['mean_psd', 'variation']:
            # add columns with data
            X_df[f'lfp_mean_{band}_{ft}'] = np.mean(
                [X_df[f'lfp_left_{band}_{ft}'],
                X_df[f'lfp_right_{band}_{ft}']], axis=0
            )
            # drop unilat lfp columns
            for s in ['left', 'right']: X_df = X_df.drop(labels=[f'lfp_{s}_{band}_{ft}'], axis=1)
        
        # drop imagniary coh columns
        X_df = X_df.drop(labels=[f'imag_coh_STN_STN_{band}'], axis=1)

    # take average over broad gamma
    # select all columns containing single gamma bands
    gamma_cols = [f for f in X_df.keys() if 'gamma1' in f]

    for col in gamma_cols:
        # add mean gammaBroad and drop single gamma cols
        X_df[col.replace('gamma1', 'gammaBroad')] = np.mean(
                [X_df[col],
                X_df[col.replace('gamma1', 'gamma2')],
                X_df[col.replace('gamma1', 'gamma3')]], axis=0
            )
        # drop unilat lfp columns
        X_df = X_df.drop(labels=[col,
                                    col.replace('gamma1', 'gamma2'),
                                    col.replace('gamma1', 'gamma3')], axis=1)
    
    return X_df.values, X_df.keys()

In [ ]:
# merge bilat STN features, merge broad gamma features
for label in X_all.keys():
    X_all[label], ft_names[label] = convert_features(X_all[label], ft_names[label])

for label in dataDicts.keys():
    
    print(label, X_all[label].shape, y_all_binary[label].shape)

print(ft_names['extVal'])

Explore movement splitting based on clustering

- Cluster has issues to differentiate small movements and rest, for now pragmatic -0.5 as cut off

In [ ]:
# plt.hist(acc_all['extVal'], bins=np.arange(-1, 4, 0.05))

# plt.show()

In [ ]:
# from sklearn.cluster import KMeans
# from sklearn.mixture import GaussianMixture  # better for skewed data

# # # KMeans clustering into 2 groups
# # kmeans = KMeans(n_clusters=2, random_state=0)
# # k_labels = kmeans.fit_predict(acc_all['extVal'].reshape(-1, 1))

# # # Optional: identify which label corresponds to rest vs movement
# # # Rest is the cluster with the lower mean RMS
# # cluster_means = [acc_all['extVal'][k_labels == i].mean() for i in range(2)]
# # rest_label = np.argmin(cluster_means)
# # movement_label = 1 - rest_label


# # Fit Gaussian Mixture Model with 2 components
# rms_values = v
# # nonlinear transformation on rms bcs of skewedness of data
# rms_values = np.sign(rms_values) * (np.abs(rms_values) ** 1.5)

# gmm = GaussianMixture(n_components=3, random_state=0,)
# gmm_labels = gmm.fit_predict(rms_values.reshape(-1, 1))

# # Identify rest vs movement based on component means
# gmm_means = gmm.means_.flatten()
# rest_label = np.argmin(gmm_means)
# movement_label = 1 - rest_label

In [ ]:
# lab='extVal'
# plt.scatter(ft_times_all[lab], y_all_binary[lab],
#             s=10, alpha=.5, label='LID',)
# plt.scatter(ft_times_all[lab], acc_all[lab],
#             s=10, alpha=.5, label='acc-rms',)

# # plt.scatter(ft_times_all[lab], k_labels + 3,
# #             s=10, alpha=.3, label='k-cluster',)
# plt.scatter(ft_times_all[lab], gmm_labels + 3,
#             s=10, alpha=.3, label='cluster labels',)


# plt.legend()
# plt.show()

In [ ]:
# lab='train'
# plt.scatter(ft_times_all[lab], y_all_binary[lab],
#             s=10, alpha=.5, label='LID',)
# plt.scatter(ft_times_all[lab], acc_all[lab],
#             s=10, alpha=.3, label='acc-rms',)

# plt.legend()
# plt.show()

## 3) Prediction

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import gpboost as gpb

import joblib

# performance
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, plot_confusion_matrix, ConfusionMatrixDisplay,
    auc, roc_curve, RocCurveDisplay
)

In [ ]:
import lfpecog_predict.predict_helpers as predHelpers
import lfpecog_plotting.plot_pred_standards as plotPred

In [ ]:
ONLY_GAMMA = True

# Model name and Path to store or load
model_name = 'extVal_lda01'

if ONLY_GAMMA: model_name = 'extVal_lda02_onlyGamma'

model_path = os.path.join(extVal_path, f'{model_name}.pkl')


# INCLUDE ALL
X_train = X_all['train']
y_train = y_all_binary['train']

X_test = X_all['extVal']
y_true_test = y_all_binary['extVal']

times_test = ft_times_all['extVal']


# ONLY-GAMMA MODEL
if ONLY_GAMMA:
    gamma_idx = ['gamma' in f for f in ft_names['train']]  # select gamma features
    print(ft_names['extVal'][gamma_idx])
    X_train = X_train[:, gamma_idx]
    X_test = X_test[:, gamma_idx]


# Train LDA
lda = LDA()
lda.fit(X_train, y_train)

# # Save model to disk
# joblib.dump(lda, model_path)


In [ ]:
# Load saved model
lda_loaded = joblib.load(model_path)
print(f'Model: {model_path} loaded')


# Predict
y_pred = lda_loaded.predict(X_test)

# predict probabilities
y_proba = lda_loaded.predict_proba(X_test)

In [ ]:
def plot_ext_validation(times, y_true, y_pred, y_probas,
                        INCL_ECOG, ONLY_GAMMA, MOVE_AWARE,
                        acc_sig=[],
                        noLID_col = 'green', LID_col = 'purple',
                        fs = 14,):
    src = 'STN'
    if INCL_ECOG: src += 'ECOG'
    fname = f'extVal_Lda_{src}_allEpochs'
    if MOVE_AWARE: fname = fname.replace('allEpochs', 'moveAware')
    if ONLY_GAMMA: fname += '_onlyGamma'


    fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                            gridspec_kw={'width_ratios': [2, 1]})

    # plot y-predictions, y-probabilities, y-true over time
    acc_score = accuracy_score(y_true=y_true, y_pred=y_pred)

    axes[0].scatter(times, y_probas[:, 0],
                    color=noLID_col, s=10, alpha=.5,
                    label='pred-proba-1',)
    axes[0].scatter(times, y_probas[:, 1],
                    color=LID_col, s=10, alpha=.5,
                    label='pred-proba-2',)
    
    # fill background for true LID state
    axes[0].fill_between(x=times, where=y_true==0,
                        y1=np.zeros(len(times)),
                        y2=np.ones(len(times)),
                        edgecolor=noLID_col, facecolor=noLID_col,
                        alpha=.25, label='true no LID',)
    axes[0].fill_between(x=times, where=y_true==1,
                        y1=np.zeros(len(times)),
                        y2=np.ones(len(times)),
                        edgecolor=LID_col, facecolor=LID_col,
                        alpha=.25, label='true LID',)
    
    # plot actual predictions
    axes[0].scatter(times, y_pred, color='orange',
                    label=f'Pred, accuracy: {np.round(acc_score, 2)}',)

    # plot acc signal
    if len(acc_sig) > 0:
        acc_ax = axes[0].twinx()
        acc_ax.plot(times, acc_sig, lw=2, color='gray', alpha=.5,)
        acc_ax.set_ylabel('Accelerometer vector (z-score)', fontsize=fs,)
        fname += '_acc'

    axes[0].set_xlabel('Time (min. vs L-DOPA intake)', size=fs,)
    axes[0].set_yticks([0, 1], size=fs,)
    axes[0].set_yticklabels(['No LID', 'LID'], size=fs,)

    axes[0].legend(ncol=3, bbox_to_anchor=[.0, 1.25],
                loc='upper left', fontsize=fs-2,
                frameon=False, )


    # plot AUROC
    fpr, tpr, _ = roc_curve(y_true, y_probas[:, 1])
    auc_score = round(auc(fpr, tpr), 2)

    axes[1].plot(fpr, tpr, c='darkgreen', lw=2,
            label=f'ROC, AUC: {auc_score}',
    )

    axes[1].plot([0, 1], [0, 1], lw=3,  c='orange', label='Chance level')

    axes[1].set_xlabel('False Positive Rate', fontsize=fs,)  #  weight='bold',
    axes[1].set_ylabel('True Positive Rate', fontsize=fs, )
    # axes[1].set_title('LID prediction - AUROC', fontsize=fs)

    axes[1].legend(frameon=False, fontsize=fs,
                loc='lower right',)

    for ax in axes:
        ax.tick_params(axis='both', labelsize=fs,)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()

    plt.savefig(os.path.join(figpath, 'prediction',
                            'ext_validation', fname),
                facecolor='w', dpi=300,)


    plt.show()


In [ ]:
plot_ext_validation(
    times=times_test,
    y_true=y_true_test,
    y_pred=y_pred,
    y_probas=y_proba,
    INCL_ECOG=INCL_ECOG,
    ONLY_GAMMA=ONLY_GAMMA,
    MOVE_AWARE=False,
    acc_sig=acc_all['extVal'],
)

Including move split

In [ ]:
ONLY_GAMMA = True

# SPLIT movement
MOVE_CUT = -.5
MOVE_SPLIT_train = acc_all['train'] > MOVE_CUT  # True for rel movement
MOVE_SPLIT_test = acc_all['extVal'] > MOVE_CUT  # True for rel movement
print(f'total samples: {len(MOVE_SPLIT_test)}; '
      f'test-no-move, n = {sum(~MOVE_SPLIT_test)}; '
      f'test-move, n = {sum(MOVE_SPLIT_test)} '
      f'({round(sum(MOVE_SPLIT_test) / len(MOVE_SPLIT_test) * 100)} %)')


# Model A: no move
model_name = 'extVal_lda01rest'
if ONLY_GAMMA: model_name = 'extVal_lda02rest_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')  # Model Path to store or load

X_train_A = X_all['train'][~MOVE_SPLIT_train]
y_train_A = y_all_binary['train'][~MOVE_SPLIT_train]

X_test_A = X_all['extVal'][~MOVE_SPLIT_test]
y_true_test_A = y_all_binary['extVal'][~MOVE_SPLIT_test]
times_test_A = ft_times_all['extVal'][~MOVE_SPLIT_test]


# ONLY-GAMMA MODEL
if ONLY_GAMMA:
    gamma_idx = ['gamma' in f for f in ft_names['train']]  # select gamma features
    print(ft_names['extVal'][gamma_idx])
    X_train_A = X_train_A[:, gamma_idx]
    X_test_A = X_test_A[:, gamma_idx]



# Train LDA
lda = LDA()
lda.fit(X_train_A, y_train_A)

# Save model to disk
# joblib.dump(lda, model_path)



# Model B: movement
model_name = 'extVal_lda01move'
if ONLY_GAMMA: model_name = 'extVal_lda02move_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')  # Model Path to store or load

X_train_B = X_all['train'][MOVE_SPLIT_train]
y_train_B = y_all_binary['train'][MOVE_SPLIT_train]

X_test_B = X_all['extVal'][MOVE_SPLIT_test]
y_true_test_B = y_all_binary['extVal'][MOVE_SPLIT_test]
times_test_B = ft_times_all['extVal'][MOVE_SPLIT_test]


# ONLY-GAMMA MODEL
if ONLY_GAMMA:
    gamma_idx = ['gamma' in f for f in ft_names['train']]  # select gamma features
    print(ft_names['extVal'][gamma_idx])
    X_train_B = X_train_B[:, gamma_idx]
    X_test_B = X_test_B[:, gamma_idx]

# Train LDA
lda = LDA()
lda.fit(X_train_B, y_train_B)

# Save model to disk
# joblib.dump(lda, model_path)


Load Models and Predict


In [ ]:

# Load saved model rest
model_name = 'extVal_lda01rest'
if ONLY_GAMMA: model_name = 'extVal_lda02rest_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')
lda_loaded = joblib.load(model_path)
print(f'NoMove-Model: {model_path} loaded')

# Predict
y_pred_A = lda_loaded.predict(X_test_A)
# predict probabilities
y_proba_A = lda_loaded.predict_proba(X_test_A)

In [ ]:

# Load saved model movement
model_name = 'extVal_lda01move'
if ONLY_GAMMA: model_name = 'extVal_lda02move_onlyGamma'
model_path = os.path.join(extVal_path, f'{model_name}.pkl')
lda_loaded = joblib.load(model_path)
print(f'Move-Model: {model_path} loaded')


# Predict
y_pred_B = lda_loaded.predict(X_test_B)

# predict probabilities
y_proba_B = lda_loaded.predict_proba(X_test_B)

In [ ]:
# merge and sort Prediction-results-arrays from rest and move models
y_times_AB = np.concatenate([times_test_A, times_test_B])
t_sort_idx = np.argsort(y_times_AB)   # sort on chronological-times
merged_t = y_times_AB[t_sort_idx]

merged_y_pred = np.concatenate([y_pred_A, y_pred_B])[t_sort_idx]
merged_y_proba = np.concatenate([y_proba_A, y_proba_B])[t_sort_idx]
merged_y_true_test = np.concatenate([y_true_test_A, y_true_test_B])[t_sort_idx]



Plot Movement Aware Models

In [ ]:
plot_ext_validation(
    times=merged_t,
    y_true=merged_y_true_test,
    y_pred=merged_y_pred,
    y_probas=merged_y_proba,
    INCL_ECOG=INCL_ECOG,
    ONLY_GAMMA=ONLY_GAMMA,
    MOVE_AWARE=True,
    acc_sig=acc_all['extVal'],
)

plot mean feature importances

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(18, 12))

fsize=20

sort_idx = np.argsort(np.mean(importances, axis=0))
temp_ftnames = plot_ssd_descr.readable_ftnames(ft_names)

ax.bar(np.arange(importances.shape[1]),
       np.mean(importances, axis=0)[sort_idx],)

ax.set_xticks(np.arange(len(ft_names)))
ax.set_xticklabels(np.array(temp_ftnames)[sort_idx],
                   rotation=60, ha='right', size=fsize)
ax.set_ylabel('importances (a.u.)', size=fsize + 8)
ax.set_xlabel('')

plt.tick_params(axis='both', size=fsize, labelsize=fsize+2)
plt.tight_layout()

fname = f'binaryLID_pred_ftImportances_ftsV4_lda'
# plt.savefig(os.path.join(figpath, 'prediction', fname),
#             facecolor='w', dpi=300,)

plt.close()

Conf Matrix

In [ ]:
importlib.reload(plotPred)


# show metrics summary
print(classification_report(y_true_all, y_pred_all))

# show confusion matrix
cm = confusion_matrix(y_true_all, y_pred_all)
cm_figname = 'Group_LID_Pred_LDA_powCoh_confMatrix'
# plotPred.plot_confMatrix(cm, fig_path=figpath, fig_name=cm_figname,
#                          to_show=False, to_save=True)

# show Receiver Operator Cruve
fpr, tpr, _ = roc_curve(y_true_all, y_pred_conf_all,)
auc_score = auc(fpr, tpr)
acc_score = accuracy_score(y_true_all, y_pred_all)
print(f'AUC: {round(auc_score, 3)}, Accuracy: {round(acc_score, 3)}')
# roc_display = RocCurveDisplay(fpr=fpr, tpr=tpr).plot()
